# Union and Dedup Notebook

This notebook consumes dataset-level standardized TSV outputs and performs the final corpus assembly stage: **union + deduplication + export**.

## Expected upstream inputs
- `outputs/preprocessing/01_hatexplain_standardized.tsv`
- `outputs/preprocessing/02_mhs_standardized.tsv`
- `outputs/preprocessing/03_elsherief_standardized.tsv` (optional, controlled by config)

## What this notebook does
1. Loads standardized per-dataset outputs.
2. Ensures required columns are present and normalized.
3. Unions datasets into a single table.
4. Deduplicates with ID-first strategy, then text-key fallback.
5. Saves final outputs (`04_union_primary.tsv`, `04_dedup_primary.tsv`, and `04_union_dedup_summary.tsv`).

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List
import re

import pandas as pd

# Resolve workspace root robustly for notebook execution in multiple contexts.
WORKDIR = Path.cwd()
if not (WORKDIR / 'outputs').exists():
    WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class UnionConfig:
    # Include or exclude ElSherief in final union.
    include_elsherief: bool = True

    # Input standardized files from the dataset-specific notebooks.
    hatexplain_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '01_hatexplain_standardized.tsv'
    mhs_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '02_mhs_standardized.tsv'
    elsherief_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '03_elsherief_standardized.tsv'

    # Final union/dedup outputs.
    union_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_union_primary.tsv'
    dedup_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_dedup_primary.tsv'
    summary_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_union_dedup_summary.tsv'

cfg = UnionConfig()
cfg.union_output.parent.mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
REQUIRED_COLUMNS = [
    'post_id',
    'text',
    'raw_label',
    'binary_hate',
    'targets',
    'dataset',
    'text_dedup_key',
]


def normalize_text_for_dedup(text: str) -> str:
    """Create a lowercase, whitespace-normalized text key for dedup fallback.

    Near-duplicate policy: only lowercase + whitespace normalization is applied.
    Posts differing only by URL, punctuation, or encoding are treated as distinct.
    This is intentionally conservative to avoid false merges. Documented in notebook.
    """
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.strip().lower())


def read_standardized_tsv(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load one standardized TSV and enforce minimum schema guarantees."""
    if not path.exists():
        raise FileNotFoundError(f'Missing standardized file for {dataset_name}: {path}')

    df = pd.read_csv(path, sep='\t', low_memory=False)

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ''

    keep_columns = REQUIRED_COLUMNS + [col for col in ['n_annotations'] if col in df.columns]
    df = df[keep_columns].copy()

    df['post_id'] = df['post_id'].astype(str).replace('nan', '')
    df['text'] = df['text'].astype(str)
    df['dataset'] = dataset_name

    missing_key = df['text_dedup_key'].isna() | (df['text_dedup_key'].astype(str).str.strip() == '')
    if missing_key.any():
        df.loc[missing_key, 'text_dedup_key'] = df.loc[missing_key, 'text'].apply(normalize_text_for_dedup)

    df['binary_hate'] = pd.to_numeric(df['binary_hate'], errors='coerce').astype('Int64')

    return df


def resolve_cross_dataset_conflicts(union_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Detect posts with the same text key that appear in multiple datasets with differing labels.

    Resolution policy: take the conservative (more hateful) label — max(binary_hate).
    Rationale: for hate-speech benchmarking false negatives are costlier than false positives,
    and this rule is dataset-load-order-independent (unlike keep-first).

    Returns:
        union_df_resolved: union_df with binary_hate resolved for conflicting keys,
            plus boolean 'label_conflict' and string 'conflict_datasets' columns.
        conflict_log: one row per conflicting text_dedup_key with original label spread.
    """
    # Identify keys that appear in more than one dataset.
    key_datasets = union_df.groupby('text_dedup_key')['dataset'].nunique()
    cross_keys = key_datasets[key_datasets > 1].index

    conflict_rows = []
    resolved_df = union_df.copy()
    resolved_df['label_conflict'] = False
    resolved_df['conflict_datasets'] = pd.NA

    for key in cross_keys:
        mask = resolved_df['text_dedup_key'] == key
        subset = resolved_df.loc[mask]
        labels = subset['binary_hate'].dropna().unique()
        datasets = sorted(subset['dataset'].unique())

        if len(labels) > 1:
            # Genuine conflict: apply conservative label to all rows sharing this key.
            conservative = int(subset['binary_hate'].max())
            resolved_df.loc[mask, 'binary_hate'] = conservative
            resolved_df.loc[mask, 'label_conflict'] = True
            resolved_df.loc[mask, 'conflict_datasets'] = '|'.join(datasets)

            conflict_rows.append({
                'text_dedup_key': key,
                'datasets': '|'.join(datasets),
                'original_labels': str(dict(zip(subset['dataset'], subset['binary_hate']))),
                'resolved_label': conservative,
            })

    conflict_log = pd.DataFrame(conflict_rows)
    return resolved_df, conflict_log


def deduplicate_union(union_df: pd.DataFrame) -> pd.DataFrame:
    """Apply ID-first dedup, then fallback text-key dedup.

    Cross-dataset label conflicts must be resolved before calling this function
    (see resolve_cross_dataset_conflicts). After resolution, keep-first is safe
    because all rows sharing a text key have the same binary_hate value.
    """
    # Pass 1: keep first row for every explicit post ID.
    with_id = union_df[union_df['post_id'].notna() & (union_df['post_id'].astype(str).str.strip() != '')]
    with_id = with_id.drop_duplicates(subset=['post_id'], keep='first')

    # Pass 2: among rows not retained in pass 1, dedup by normalized text key.
    without_id = union_df[~union_df.index.isin(with_id.index)]
    without_id = without_id.drop_duplicates(subset=['text_dedup_key'], keep='first')

    # Final pass: one row per text key globally to remove cross-source repeats.
    dedup_df = pd.concat([with_id, without_id], ignore_index=True)
    dedup_df = dedup_df.drop_duplicates(subset=['text_dedup_key'], keep='first')

    return dedup_df


In [ ]:
# Load mandatory standardized datasets.
hx_std = read_standardized_tsv(cfg.hatexplain_tsv, dataset_name='hatexplain')
mhs_std = read_standardized_tsv(cfg.mhs_tsv, dataset_name='mhs')

parts: List[pd.DataFrame] = [hx_std, mhs_std]

if cfg.include_elsherief:
    if cfg.elsherief_tsv.exists():
        els_std = read_standardized_tsv(cfg.elsherief_tsv, dataset_name='elsherief')
        parts.append(els_std)
    else:
        print('ElSherief inclusion is enabled, but standardized file is missing; continuing without it.')

# Union all selected standardized datasets.
union_df = pd.concat(parts, ignore_index=True)

# Resolve cross-dataset label conflicts before dedup.
union_df, conflict_log = resolve_cross_dataset_conflicts(union_df)
n_conflict_keys = len(conflict_log)
n_conflict_rows = int(union_df['label_conflict'].sum())

if n_conflict_keys:
    print(f'Cross-dataset label conflicts resolved (conservative/hateful label applied):')
    display(conflict_log)
else:
    print('No cross-dataset label conflicts detected.')

# Deduplicate. Label conflicts are already resolved so keep-first is order-independent.
dedup_df = deduplicate_union(union_df)

# Build summary table.
summary_df = pd.DataFrame([
    {'metric': 'hatexplain_rows', 'value': len(hx_std)},
    {'metric': 'mhs_rows', 'value': len(mhs_std)},
    {'metric': 'elsherief_rows_included', 'value': int((dedup_df['dataset'] == 'elsherief').sum()) if 'dataset' in dedup_df.columns else 0},
    {'metric': 'union_rows', 'value': len(union_df)},
    {'metric': 'cross_dataset_conflict_keys', 'value': n_conflict_keys},
    {'metric': 'cross_dataset_conflict_rows_in_union', 'value': n_conflict_rows},
    {'metric': 'dedup_rows', 'value': len(dedup_df)},
    {'metric': 'dedup_rows_with_label_conflict', 'value': int(dedup_df['label_conflict'].sum())},
])

union_df.to_csv(cfg.union_output, sep='\t', index=False)
dedup_df.to_csv(cfg.dedup_output, sep='\t', index=False)
summary_df.to_csv(cfg.summary_output, sep='\t', index=False)

print('Saved union output to:', cfg.union_output)
print('Saved dedup output to:', cfg.dedup_output)
print('Saved summary output to:', cfg.summary_output)
display(summary_df)
display(dedup_df.head(5))


## Next Step: Target Label Analysis

Target-label standardization, groupwise label analysis, and post-union low-group dropping have been moved to:

- `data_preprocessing/05_target_label_analysis_and_filtering.ipynb`

This notebook now focuses only on union + dedup output generation.

---

## Design Decisions

### Cross-dataset label conflicts

**Problem:** The same post text can appear in multiple source datasets with a different `binary_hate` label (e.g., HateXplain labels it 0, MHS labels it 1).

**Observed scale (with current three-dataset union):**
- 8 text keys appear in 2+ datasets.
- 4 of those have conflicting `binary_hate` values.

**Previous behavior:** "keep first occurrence" — the surviving label depended on which dataset was concatenated first (HateXplain → MHS → ElSherief), making results load-order-dependent and not reproducible across dataset subsets.

**Decision: conservative (hateful-wins) label, logged.**
- `binary_hate` for all rows sharing a conflicting text key is set to `max(binary_hate)` across the contributing datasets.
- A boolean `label_conflict` column and a `conflict_datasets` column are added to the union and dedup outputs.
- A `conflict_log` table is printed at runtime for inspection.
- **Rationale:** In a hate-speech benchmarking context, false negatives (letting hate content be labeled non-hate) are a more serious evaluation error than false positives. The `max` rule is dataset-load-order-independent. The `label_conflict` flag allows downstream analyses to exclude or separately analyze these ambiguous posts if desired.

---

### Near-duplicate handling

**Problem:** The `text_dedup_key` is built by lowercasing and collapsing whitespace only. Two posts that are semantically identical but differ by a trailing URL, punctuation normalization (e.g., curly vs. straight quotes), or encoding (e.g., `&amp;` vs. `&`) will not be matched as duplicates.

**Observed scale:** At current dataset sizes this is not quantified, but the source datasets are relatively curated social-media collections where URL-appended variants are uncommon.

**Decision: accept exact-text matching; document the limitation.**
- No additional normalization (URL stripping, punctuation folding, Unicode normalization) is applied to `text_dedup_key`.
- **Rationale:** More aggressive normalization risks false merges — two genuinely different posts that happen to match after stripping. Exact-text matching is a conservative, reproducible baseline. Any near-duplicate leakage is likely small relative to the 81 K corpus and symmetric across datasets, so it should not materially affect disparity estimates.
- If future work requires tighter dedup (e.g., for contamination analysis), a URL-stripped key can be added as an additional pass without changing the primary dedup key.
